Загружаме датасет

In [2]:
from sqlalchemy import create_engine
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

def create_connection():

    load_dotenv()
    host = os.environ.get('DB_DESTINATION_HOST')
    port = os.environ.get('DB_DESTINATION_PORT')
    db = os.environ.get('DB_DESTINATION_NAME')
    username = os.environ.get('DB_DESTINATION_USER')
    password = os.environ.get('DB_DESTINATION_PASSWORD')

    conn = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{db}')
    return conn

conn = create_connection()
data = pd.read_sql("select * from flats_price", conn)
data

,id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,target
0,16937,20493,12,8.0,47.000000,3,False,False,75.000000,21000000,2007,2,55.774887,37.707790,2.80,179,14,True,21000000
1,16938,8742,2,8.6,20.000000,1,False,False,38.000000,12790000,1970,1,55.758003,37.542652,2.75,218,14,True,12790000
2,16939,11960,2,6.0,19.000000,1,False,False,33.000000,7600000,1977,4,55.605545,37.737003,2.50,191,12,True,7600000
3,16940,24382,3,11.0,15.000000,1,False,False,37.000000,7500000,2019,2,55.592587,37.206890,2.80,446,17,True,7500000
4,16941,8536,5,6.0,24.100000,2,False,False,39.299999,10590000,1969,6,55.625599,37.657951,2.64,81,12,True,10590000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141357,118224,13927,1,9.0,19.000000,1,False,False,37.799999,11500000,1983,4,55.811291,37.633957,2.64,203,17,True,11500000
141358,118225,13927,1,9.0,19.000000,1,False,False,37.799999,11500000,1983,4,55.811291,37.633957,2.64,203,17,True,11500000
141359,118226,10080,6,0.0,0.000000,2,False,False,44.099998,12600000,1972,4,55.846592,37.444603,2.64,216,9,True,12600000
141360,118227,7876,1,5.8,41.099998,3,False,False,58.299999,12300000,1968,4,55.862465,37.508392,2.64,179,5,False,12300000


Дубликаты

Дубликаты по id

In [3]:
is_duplicated_id = data.duplicated(subset=['id'], keep=False)
print(sum(is_duplicated_id)) 

0


In [4]:
feature_cols = data.drop(['id'], axis=1).columns.tolist()
is_duplicated_features = data.duplicated(subset = feature_cols, keep=False)
print(sum(is_duplicated_features))

17425


0 дубликатов по id и 17425 полных дубликатов по признакам. Их доля значительна, поэтому оставим ориганал(оригинал будем считать первую попавшуюся).

In [5]:
def remove_duplicates(data):
    feature_cols = data.columns.drop('id').tolist()
    is_duplicated_features = data.duplicated(subset=feature_cols, keep='first')
    data = data[~is_duplicated_features].reset_index(drop=True)
    return data 
remove_duplicates(data)

,id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,target
0,16937,20493,12,8.0,47.000000,3,False,False,75.000000,21000000,2007,2,55.774887,37.707790,2.80,179,14,True,21000000
1,16938,8742,2,8.6,20.000000,1,False,False,38.000000,12790000,1970,1,55.758003,37.542652,2.75,218,14,True,12790000
2,16939,11960,2,6.0,19.000000,1,False,False,33.000000,7600000,1977,4,55.605545,37.737003,2.50,191,12,True,7600000
3,16940,24382,3,11.0,15.000000,1,False,False,37.000000,7500000,2019,2,55.592587,37.206890,2.80,446,17,True,7500000
4,16941,8536,5,6.0,24.100000,2,False,False,39.299999,10590000,1969,6,55.625599,37.657951,2.64,81,12,True,10590000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132241,118222,8926,1,0.0,12.000000,1,False,False,15.000000,3250000,1970,4,55.720055,37.811520,2.64,144,9,True,3250000
132242,118223,13927,1,9.0,19.000000,1,False,False,37.799999,15200000,1983,4,55.811291,37.633957,2.64,203,17,True,15200000
132243,118224,13927,1,9.0,19.000000,1,False,False,37.799999,11500000,1983,4,55.811291,37.633957,2.64,203,17,True,11500000
132244,118226,10080,6,0.0,0.000000,2,False,False,44.099998,12600000,1972,4,55.846592,37.444603,2.64,216,9,True,12600000


### Пропуски

In [11]:
data.isnull().sum()

id                   0
building_id          0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
target               0
dtype: int64

Пропусков нет, но могут появится в будущем, выгоднее будет их тогда удалить, так как их число будет мало по сравнению с выборокой.

In [7]:
def remove_missing_values(data: pd.DataFrame) -> pd.DataFrame:
    return data.dropna().reset_index(drop=True)
remove_missing_values(data)


,id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,target
0,16937,20493,12,8.0,47.000000,3,False,False,75.000000,21000000,2007,2,55.774887,37.707790,2.80,179,14,True,21000000
1,16938,8742,2,8.6,20.000000,1,False,False,38.000000,12790000,1970,1,55.758003,37.542652,2.75,218,14,True,12790000
2,16939,11960,2,6.0,19.000000,1,False,False,33.000000,7600000,1977,4,55.605545,37.737003,2.50,191,12,True,7600000
3,16940,24382,3,11.0,15.000000,1,False,False,37.000000,7500000,2019,2,55.592587,37.206890,2.80,446,17,True,7500000
4,16941,8536,5,6.0,24.100000,2,False,False,39.299999,10590000,1969,6,55.625599,37.657951,2.64,81,12,True,10590000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141357,118224,13927,1,9.0,19.000000,1,False,False,37.799999,11500000,1983,4,55.811291,37.633957,2.64,203,17,True,11500000
141358,118225,13927,1,9.0,19.000000,1,False,False,37.799999,11500000,1983,4,55.811291,37.633957,2.64,203,17,True,11500000
141359,118226,10080,6,0.0,0.000000,2,False,False,44.099998,12600000,1972,4,55.846592,37.444603,2.64,216,9,True,12600000
141360,118227,7876,1,5.8,41.099998,3,False,False,58.299999,12300000,1968,4,55.862465,37.508392,2.64,179,5,False,12300000


### Выбросы

Найдем выбросы метод IQR, с threshold = 1.5

In [8]:
num_cols = data.select_dtypes(['float']).columns
threshold = 1.5
potential_outliers = pd.DataFrame()

for col in num_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    margin = IQR * threshold
    lower = Q1 - margin
    upper = Q3 + margin
    potential_outliers[col] = ~data[col].between(lower, upper)

outliers = potential_outliers.any(axis=1)

print(potential_outliers.sum())
print(outliers.sum())
print(outliers.mean())

kitchen_area      8076
living_area       4962
total_area        7839
latitude           111
longitude         1975
ceiling_height    9989
dtype: int64
20674
0.14624863824790255


Выбросов оказалось много 14.6% от выборки, если всех их удаить, потеряем немалую часть данных. Поменяем значение threshold

In [9]:
for threshold in [1.5, 2, 2.5, 3]:
    potential_outliers = pd.DataFrame()
    for col in num_cols:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        margin = IQR * threshold
        lower = Q1 - margin
        upper = Q3 + margin
        potential_outliers[col] = ~data[col].between(lower, upper)

    outliers = potential_outliers.any(axis=1)
    print(threshold)
    print(outliers.sum())
    print(outliers.mean())

1.5
20674
0.14624863824790255
2
14418
0.10199346358993223
2.5
11629
0.08226397475983645
3
7066
0.04998514452257325


При threshold = 3, выбросами было призанано примерно 5%, поэтому данный порог был выбран как менее агрессивный.

In [10]:
def remove_outliers(data: pd.DataFrame) -> pd.DataFrame:
    potential_outliers = pd.DataFrame()
    num_cols = data.select_dtypes(['float']).columns
    threshold = 3
    for col in num_cols:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        margin = IQR * threshold
        lower = Q1 - margin
        upper = Q3 + margin
        potential_outliers[col] = ~data[col].between(lower, upper)
    outliers = potential_outliers.any(axis=1)
    data = data[~outliers]
    return data
data = remove_duplicates(data)
data = remove_missing_values(data)
data = remove_outliers(data)
data.shape

(125768, 19)